# HASA: hands-on demo

**HASA: Subnet Allocation for Compute-Constrained Model-Heterogeneous Federated Learning**
Amir Hossein Shahdadian, Ahmed M. Abdelmoniem, Mahdi Taheri, Samira Nazari, Christian Herglotz · IEEE EDGE 2026

Paper: [arXiv:2606.07621](https://arxiv.org/abs/2606.07621) · Code: [github.com/Reliable-and-Secure-Systems/HASA](https://github.com/Reliable-and-Secure-Systems/HASA)

Each cell runs one claim of the paper.

| Cell | What it shows | Paper |
|---|---|---|
| 1 | The benchmark: seven Medium publications as clients | Sec. V-A |
| 2 | Train-only heterogeneity score $h_i = \mathrm{JSD}(p_i \,\|\, p_g)$ | Sec. IV-A, Table III |
| 3 | HASA allocation (Algorithm 1) under a fixed budget $B = 0.5$ | Sec. IV-B, Algorithm 1 |
| 4 | Slimmable LSTM: width $r$ keeps the first $\lfloor r \cdot 256 \rfloor$ hidden units | Sec. III-A, V-A |
| 5 | Federated training: HASA vs Uniform under the same budget | Sec. VI-A |
| 6 | This run next to the published 10-seed results | Fig. 3, Table III |

Cells 1 to 4 take seconds. Cell 5 trains two 50-round federated runs, so use a GPU runtime in Colab (*Runtime → Change runtime type → T4 GPU*).

## 1. The benchmark

In [ ]:
import os
if os.path.exists("../hasa"):          # opened from notebooks/ inside the repository
    os.chdir("..")
elif not os.path.exists("hasa"):       # Google Colab: fetch the code
    !git clone -q https://github.com/Reliable-and-Secure-Systems/HASA.git
    os.chdir("HASA")
!pip -q install --disable-pip-version-check demoji==1.1.0

import numpy as np, pandas as pd
from hasa import load_benchmark, heterogeneity_scores, hasa_allocation, run_policy, get_device

device = get_device()
print("device:", device)
bench = load_benchmark("data/medium_data.csv")

## 2. Train-only heterogeneity score

Each client's token distribution $p_i$ is compared with the pooled distribution $p_g$ of all clients' training titles, using the Jensen–Shannon divergence. Only training titles are used, never validation or test data.

In [ ]:
scores = heterogeneity_scores(bench)

clients = pd.DataFrame({"client": bench.client_names,
                        "train sequences": bench.client_data_sizes,
                        "JSD h_i": np.round(scores, 3)})
clients

## 3. HASA allocation: Algorithm 1

Rank the scores, map the ranks linearly to $[r_{\min}, r_{\max}] = [0.2, 0.8]$, then rescale and clip until the size-weighted mean width equals the budget $B = 0.5$. The three lines after `p = ...` are Algorithm 1; the `assert` checks them against the library implementation.

In [ ]:
n = np.array(bench.client_data_sizes, dtype=float)
p = n / n.sum()                                          # client weights n_i / N
r = 0.2 + 0.6 * np.argsort(np.argsort(scores)) / (len(scores) - 1)   # rank-normalise, map to [0.2, 0.8]
for t in range(2):                                       # scale to the budget, clip to the bounds
    r = np.clip(r * 0.5 / (p @ r), 0.2, 0.8)
assert np.allclose(r, hasa_allocation(bench.client_data_sizes, scores))

clients["HASA width (%)"] = np.round(100 * r, 1)
clients["Uniform width (%)"] = 50.0
print(f"size-weighted mean width:  HASA {p @ r:.3f}   Uniform 0.500   (budget B = 0.5)")
clients

## 4. Slimmable LSTM

One shared LSTM (embedding 128, hidden 256). A client with width $r$ trains only the first $\lfloor r \cdot 256 \rfloor$ hidden units: the code slices the gate rows, the recurrent block and the output columns, so a narrow subnet really computes less.

In [ ]:
import inspect
from hasa.model import forward_subnet_with_indices
print(inspect.getsource(forward_subnet_with_indices))

E, H = 128, 256
units = [max(16, int(H * w)) for w in r]
clients["hidden units"] = units
clients["LSTM compute vs full (%)"] = [round(100 * (4*a*E + 4*a*a) / (4*H*E + 4*H*H), 1) for a in units]
clients[["client", "HASA width (%)", "hidden units", "LSTM compute vs full (%)"]]

## 5. Federated training: HASA vs Uniform

Same data, same initial model, same seed and the same compute budget; only the client-to-width assignment changes. This is the paper's protocol: 50 FedAvg rounds, 1 local epoch per round, Adam (learning rate 0.001), batch 64. Each client is evaluated with the global model on its own test split, at its own width.

In [ ]:
runs = {}
for name, policy in [("Uniform", "uniform_50"), ("HASA", "difficulty_only")]:
    runs[name] = run_policy(bench, scores, policy, device, seed=42)

## 6. Measured next to published

The paper reports the mean over 10 seeds; a single seed moves by a few tenths of a point, so compare the trend.

In [ ]:
import matplotlib.pyplot as plt

PAPER = {"Uniform": (13.82, 11.47, 12.04), "HASA": (14.32, 11.99, 12.52)}        # Fig. 3, 10 seeds
PAPER_DELTA = [0.45, 2.18, 0.47, 1.99, -0.76, -0.26, -0.91]                        # Table III, HASA - Uniform
COLORS = {"Uniform": "#4B4A9A", "HASA": "#C33D6C"}
INK, MUTED = "#222238", "#6F7182"

rows = []
for name, h in runs.items():
    rows.append([name, *(round(100 * h[k][-1], 2) for k in ("test_mean", "test_worst", "test_p10")), *PAPER[name]])
table = pd.DataFrame(rows, columns=["policy", "mean", "worst", "P10", "paper mean", "paper worst", "paper P10"])
display(table)

plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.15]})

from matplotlib.lines import Line2D
metrics, rows_y = ["mean", "worst", "P10"], np.arange(3)[::-1]
for dy, source in [(0.17, "run"), (-0.17, "paper")]:
    vals = {n: (table.loc[table.policy == n, metrics].values[0] if source == "run" else np.array(PAPER[n]))
            for n in ("Uniform", "HASA")}
    ax1.hlines(rows_y + dy, vals["Uniform"], vals["HASA"], color=MUTED, lw=1.5, zorder=1)
    for n in ("Uniform", "HASA"):
        ax1.scatter(vals[n], rows_y + dy, s=110, zorder=3, linewidths=2, edgecolors=COLORS[n],
                    color=COLORS[n] if source == "run" else "white")
ax1.set_yticks(rows_y, ["Mean", "Worst client", "P10"])
ax1.set_xlabel("client test accuracy (%)")
ax1.grid(axis="x", color="#E6E9F2")
ax1.set_axisbelow(True)
dot = lambda face, edge: Line2D([], [], marker="o", ls="", ms=9, mfc=face, mec=edge, mew=2)
ax1.legend([dot(COLORS["Uniform"], COLORS["Uniform"]), dot(COLORS["HASA"], COLORS["HASA"]),
            dot(MUTED, MUTED), dot("white", MUTED)],
           ["Uniform", "HASA", "this run (seed 42)", "paper (10 seeds)"],
           frameon=False, fontsize=10, loc="upper left")
ax1.set_title("HASA vs Uniform, same budget", loc="left", color=INK)

delta = 100 * (np.array(runs["HASA"]["per_client_test"][-1]) - np.array(runs["Uniform"]["per_client_test"][-1]))
y = np.arange(len(bench.client_names))[::-1]
ax2.barh(y, delta, 0.55, color=COLORS["HASA"], label="this run")
ax2.scatter(PAPER_DELTA, y, marker="D", s=40, color=INK, zorder=3, label="paper, 10 seeds")
ax2.axvline(0, color=MUTED, lw=1)
ax2.set_yticks(y, [f"{c} ({wd:.0f}%)" for c, wd in zip(bench.client_names, 100 * r)])
ax2.set_xlabel("accuracy gain of HASA over Uniform (pp)")
ax2.grid(axis="x", color="#E6E9F2")
ax2.set_axisbelow(True)
ax2.set_ylim(-1.4, len(y) - 0.5)                         # empty strip at the bottom for the legend
ax2.legend(frameon=False, fontsize=10, loc="lower right", ncol=2)
ax2.set_title("Per client (HASA width in brackets)", loc="left", color=INK)
plt.tight_layout()
plt.show()